# AutoML comparison — FLAML vs. the Spark MLlib baseline

Same data, same temporal split, same metrics as `databricks_distributed_pipeline.ipynb`, but the model
and its hyperparameters are chosen by **AutoML** ([FLAML](https://github.com/microsoft/FLAML), an
open-source library from Microsoft Research) instead of a hand-written grid.

| | Spark baseline | This notebook |
|---|---|---|
| Model family | `GBTClassifier` only | chosen by AutoML: LightGBM, XGBoost, random forest, extra trees |
| Search | grid, 4 combinations | cost-aware search (FLAML's CFO/BlendSearch) within a time budget |
| Validation | one random 80/20 split | stratified 5-fold cross-validation (the paper used 10-fold) |
| Tuning objective | GIBA, the paper's objective | GIBA, the paper's objective |
| Class imbalance | per-row weight `n_neg / n_pos` | same weights, passed as `sample_weight` |
| Where it runs | Spark executors (distributed) | the notebook's driver (**single node**, pandas in memory) |
| Test set | 2021 hold-out | 2021 hold-out |

**Why FLAML and not Databricks AutoML?** Databricks' built-in AutoML for classification needs a
classic cluster with the ML runtime, which Free Edition (serverless only) does not offer. FLAML
installs with `%pip` and runs on serverless.

**This notebook is not distributed.** The data (about 50 MB) fits in memory, so AutoML runs on one
machine. That is the honest trade-off the comparison shows: a wider, smarter search on a single node
against a narrow grid on a distributed engine.

Run `databricks_distributed_pipeline.ipynb` first — it creates the Volume, copies the data, and writes
the baseline metrics table this notebook reads.

## 0. Install FLAML

In [0]:
%pip install -q "flaml[automl]"

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

## 1. Configuration

`TIME_BUDGET` is the AutoML search time **per hazard**, in seconds. Raise it for a deeper search.

In [0]:
CATALOG = "workspace"
SCHEMA  = "default"
VOLUME  = "thesis_data"

LABELS = ["flood_risk", "landslide_risk"]
PREFIX = {"flood_risk": ("xf", "yf"), "landslide_risk": ("xl", "yl")}

TIME_BUDGET = 300                                   # seconds of search per hazard
ESTIMATORS  = ["lgbm", "xgboost", "xgb_limitdepth", "rf", "extra_tree"]
N_SPLITS    = 5                                     # stratified k-fold CV (the paper used 10)
SEED        = 0

BASE    = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DATA    = f"{BASE}/Databases/TrainTestSplit"
RESULTS = f"{BASE}/results"
print("reading from", DATA, "| budget per hazard:", TIME_BUDGET, "s")

reading from /Volumes/workspace/default/thesis_data/Databases/TrainTestSplit | budget per hazard: 300 s


## 2. Load data

Read with pandas on the driver. Feature names are sanitized because LightGBM rejects special
characters; the 4 ID columns are dropped, leaving the same 112 features as the baseline.

In [0]:
import os, re, time, json
import numpy as np
import pandas as pd

os.makedirs(RESULTS, exist_ok=True)
ID_COLS = ["conglome", "vivienda", "hogar", "year"]

def load(label):
    xp, yp = PREFIX[label]
    def xy(split):
        X = pd.read_csv(f"{DATA}/{xp}_{split}.csv").drop(columns=ID_COLS)
        y = pd.read_csv(f"{DATA}/{yp}_{split}.csv")[label].astype(int)
        X.columns = [re.sub(r"[^0-9A-Za-z_]", "_", c) for c in X.columns]
        assert X.shape[1] == 112 and len(X) == len(y)
        return X.astype("float32"), y
    return (*xy("train"), *xy("test"))

## 3. The paper's imbalance metrics

Identical formulas to the baseline notebook, computed with NumPy at the default 0.5 probability
threshold. `giba_loss` wraps the paper's objective in the form FLAML expects for a custom metric, so
AutoML **tunes on the same objective the paper maximized**.

In [0]:
import math

def giba(tpr, tnr):
    # The paper's tuning objective, as its code computes it:
    #   imblearn make_index_balanced_accuracy(alpha=0.9, squared=True)(geometric_mean_score)
    # With that call the dominance term is zero (sensitivity and specificity are macro-averaged),
    # so the score equals G-Mean squared = sensitivity x specificity.
    return tpr * tnr

def metrics_from_counts(TP, TN, FP, FN, auc):
    tpr  = TP/(TP+FN) if TP+FN else 0.0     # sensitivity / recall
    tnr  = TN/(TN+FP) if TN+FP else 0.0     # specificity
    prec = TP/(TP+FP) if TP+FP else 0.0
    gms  = math.sqrt(tpr*tnr)
    den  = math.sqrt((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN))
    mcc  = (TP*TN - FP*FN)/den if den else 0.0
    f1   = 2*prec*tpr/(prec+tpr) if prec+tpr else 0.0
    return {"AUC": auc, "Sensitivity": tpr, "Specificity": tnr, "BalancedAcc": (tpr+tnr)/2,
            "GMean": gms, "GIBA": giba(tpr, tnr), "MCC": mcc, "Precision": prec, "F1": f1,
            "TP": TP, "TN": TN, "FP": FP, "FN": FN}

from sklearn.metrics import roc_auc_score

def imbalance_metrics(y_true, proba, threshold=0.5):
    y = np.asarray(y_true); pred = (np.asarray(proba) >= threshold).astype(int)
    TP = int(((y == 1) & (pred == 1)).sum()); TN = int(((y == 0) & (pred == 0)).sum())
    FP = int(((y == 0) & (pred == 1)).sum()); FN = int(((y == 1) & (pred == 0)).sum())
    return metrics_from_counts(TP, TN, FP, FN, float(roc_auc_score(y, proba)))

def giba_loss(X_val, y_val, estimator, labels, X_train, y_train,
              weight_val=None, weight_train=None, *args):
    # FLAML custom metric: minimize 1 - GIBA on each validation fold (0.5 threshold)
    y = np.asarray(y_val); pred = np.asarray(estimator.predict(X_val))
    tpr = ((pred == 1) & (y == 1)).sum() / max(int((y == 1).sum()), 1)
    tnr = ((pred == 0) & (y == 0)).sum() / max(int((y == 0).sum()), 1)
    score = giba(tpr, tnr)
    return 1 - score, {"giba": score, "sensitivity": tpr, "specificity": tnr}

## 4. One AutoML experiment

FLAML searches over model families and their hyperparameters until the time budget runs out, scoring
each candidate by cross-validated GIBA on the 2018–2020 training years. The winner is refit on
all training data and evaluated once on the 2021 hold-out.

In [0]:
from flaml import AutoML

def run_automl(label):
    X_train, y_train, X_test, y_test = load(label)
    n_pos, n_neg = int((y_train == 1).sum()), int((y_train == 0).sum())
    pos_weight = n_neg / n_pos
    weights = np.where(y_train == 1, pos_weight, 1.0)
    print(f"===== {label} =====")
    print("train rows:", len(y_train), "| positives:", n_pos,
          "| pos_weight:", round(pos_weight, 2), "| test rows:", len(y_test))

    automl = AutoML()
    t0 = time.time()
    automl.fit(X_train, y_train, task="classification", metric=giba_loss,
               time_budget=TIME_BUDGET, estimator_list=ESTIMATORS,
               eval_method="cv", n_splits=N_SPLITS, split_type="stratified",
               sample_weight=weights, seed=SEED, verbose=1)
    seconds = time.time() - t0

    cv_giba = 1 - automl.best_loss
    trials = len(automl.config_history)
    print("best model:", automl.best_estimator, "| CV GIBA:", round(cv_giba, 4),
          "| configurations tried:", trials, "| search time:", round(seconds, 1), "s")
    print("best config:", automl.best_config)
    print("best CV GIBA per model family:",
          {k: round(1 - v, 4) for k, v in automl.best_loss_per_estimator.items() if np.isfinite(v)})

    m = imbalance_metrics(y_test, automl.predict_proba(X_test)[:, 1])
    print(f"--- Test-set metrics ({label}, 2021 hold-out) ---")
    for k, v in m.items():
        print(f"{k:>12}: {round(v, 4) if isinstance(v, float) else v}")

    imp = pd.Series(np.asarray(automl.feature_importances_, dtype=float), index=list(automl.feature_names_in_))
    print("Top 15 features (share of the model's importance):")
    print((imp / imp.sum()).sort_values(ascending=False).head(15).round(4).to_string())

    return {"label": label, "model": f"FLAML AutoML ({automl.best_estimator})",
            "search": f"AutoML, {trials} configs, {N_SPLITS}-fold CV, {TIME_BUDGET}s budget, objective GIBA",
            "best_config": json.dumps(automl.best_config, default=float),
            "search_seconds": round(seconds, 1), "validation_giba": cv_giba, **m}

results = {}

Tried to attach usage logger `pyspark.databricks.pandas.usage_logger`, but an exception was raised: JVM wasn't initialised. Did you call it on executor side?


## 5. Floods

In [0]:
results["flood_risk"] = run_automl("flood_risk")

===== flood_risk =====
train rows: 72952 | positives: 1020 | pos_weight: 70.52 | test rows: 28163
best model: xgboost | CV GIBA: 0.6893 | configurations tried: 12 | search time: 300.4 s
best config: {'n_estimators': 21, 'max_leaves': 28, 'min_child_weight': 0.001, 'learning_rate': np.float64(0.12409433174697636), 'subsample': np.float64(0.9365144411874639), 'colsample_bylevel': np.float64(0.9660369038623221), 'colsample_bytree': np.float64(0.9906912896984851), 'reg_alpha': np.float64(0.005968336687722715), 'reg_lambda': np.float64(0.4081293963340212)}
best CV GIBA per model family: {'lgbm': np.float64(0.6692), 'xgboost': np.float64(0.6893), 'xgb_limitdepth': np.float64(0.6736), 'rf': np.float64(0.601), 'extra_tree': np.float64(0.6202)}
--- Test-set metrics (flood_risk, 2021 hold-out) ---
         AUC: 0.8871
 Sensitivity: 0.7727
 Specificity: 0.8239
 BalancedAcc: 0.7983
       GMean: 0.7979
        GIBA: 0.6366
         MCC: 0.1297
   Precision: 0.0301
          F1: 0.058
          TP:

## 6. Landslides

In [0]:
results["landslide_risk"] = run_automl("landslide_risk")

===== landslide_risk =====
train rows: 72952 | positives: 3931 | pos_weight: 17.56 | test rows: 28163
best model: lgbm | CV GIBA: 0.6804 | configurations tried: 11 | search time: 303.7 s
best config: {'n_estimators': 24, 'num_leaves': 106, 'min_child_samples': 22, 'learning_rate': np.float64(0.16249964847871098), 'log_max_bin': 8, 'colsample_bytree': np.float64(0.7413094950796925), 'reg_alpha': np.float64(0.001402029720152807), 'reg_lambda': np.float64(0.09090424151579594)}
best CV GIBA per model family: {'lgbm': np.float64(0.6804), 'xgboost': np.float64(0.6764), 'rf': np.float64(0.6561), 'extra_tree': np.float64(0.6654)}
--- Test-set metrics (landslide_risk, 2021 hold-out) ---
         AUC: 0.9116
 Sensitivity: 0.7729
 Specificity: 0.8532
 BalancedAcc: 0.8131
       GMean: 0.8121
        GIBA: 0.6595
         MCC: 0.3736
   Precision: 0.2385
          F1: 0.3646
          TP: 1222
          TN: 22681
          FP: 3901
          FN: 359
Top 15 features (share of the model's importance

## 7. Comparison with the Spark baseline

Loads the baseline table written by `databricks_distributed_pipeline.ipynb` and puts both approaches
side by side on the 2021 hold-out.

In [0]:
automl_df = pd.DataFrame([results[l] for l in LABELS if l in results])
automl_df.to_csv(f"{RESULTS}/automl_flaml_metrics.csv", index=False)
print("saved:", f"{RESULTS}/automl_flaml_metrics.csv")

baseline_path = f"{RESULTS}/spark_gbt_metrics.csv"
if os.path.exists(baseline_path):
    both = pd.concat([pd.read_csv(baseline_path), automl_df], ignore_index=True)
else:
    print("baseline table not found — run databricks_distributed_pipeline.ipynb first")
    both = automl_df

cols = ["label", "model", "AUC", "Sensitivity", "Specificity", "BalancedAcc", "GMean", "GIBA",
        "MCC", "Precision", "F1", "TP", "FN", "FP", "TN", "search_seconds"]
both = both.sort_values(["label", "model"]).reset_index(drop=True)
both.to_csv(f"{RESULTS}/comparison.csv", index=False)
pd.set_option("display.width", 250)
print(both[cols].round(4).to_string(index=False))

saved: /Volumes/workspace/default/thesis_data/results/automl_flaml_metrics.csv
         label                  model    AUC  Sensitivity  Specificity  BalancedAcc  GMean   GIBA    MCC  Precision     F1   TP  FN   FP    TN  search_seconds
    flood_risk FLAML AutoML (xgboost) 0.8871       0.7727       0.8239       0.7983 0.7979 0.6366 0.1297     0.0301 0.0580  153  45 4925 23040           300.4
    flood_risk        Spark MLlib GBT 0.8145       0.6717       0.8312       0.7515 0.7472 0.5583 0.1113     0.0274 0.0527  133  65 4720 23245           269.2
landslide_risk    FLAML AutoML (lgbm) 0.9116       0.7729       0.8532       0.8131 0.8121 0.6595 0.3736     0.2385 0.3646 1222 359 3901 22681           303.7
landslide_risk        Spark MLlib GBT 0.8734       0.8710       0.7332       0.8021 0.7991 0.6386 0.3032     0.1626 0.2740 1377 204 7093 19489           252.3


In [0]:
print(both[["label", "model", "search", "best_config"]].to_string(index=False))

         label                  model                                                     search                                                                                                                                                                                                                                                                                    best_config
    flood_risk FLAML AutoML (xgboost) AutoML, 12 configs, 5-fold CV, 300s budget, objective GIBA {"n_estimators": 21, "max_leaves": 28, "min_child_weight": 0.001, "learning_rate": 0.12409433174697636, "subsample": 0.9365144411874639, "colsample_bylevel": 0.9660369038623221, "colsample_bytree": 0.9906912896984851, "reg_alpha": 0.005968336687722715, "reg_lambda": 0.4081293963340212}
    flood_risk        Spark MLlib GBT                grid, 4 combos, 80/20 split, objective GIBA                                                                                                                                        

## Notes on reading the comparison

- **Same objective.** Both notebooks tune on GIBA at the 0.5 threshold, the objective the paper
  maximized, with the same class weights. AUC is reported as a threshold-free reference only.
- **Search effort differs.** The baseline tries 4 configurations of one model; AutoML tries many
  configurations of several models. A better AutoML score reflects the wider search, not Spark
  versus single-node.
- **Validation differs.** AutoML selects by 5-fold stratified CV; the baseline by one 80/20 split;
  the paper by 10-fold stratified CV.
- **One run each.** There are no repeated runs or confidence intervals, so small differences between
  the two approaches should not be read as significant.